# Notebook 07 — Indexation des documents RAG dans Chroma

Ce notebook charge les documents depuis `data/rag_documents/raw/`, les découpe en chunks,
les encode avec `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
et les indexe dans la base vectorielle Chroma persistée dans `data/rag_documents/chroma_db/`.

**À exécuter une seule fois** (ou après ajout de nouveaux documents).

Pipeline :
1. Charger les documents via `DocumentLoader`
2. Inspecter les chunks
3. Indexer dans Chroma via `VectorStoreManager`
4. Vérifier avec 3 requêtes test via `Retriever`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

from pathlib import Path
from config.settings import RAG_DOCUMENTS_PATH, CHROMA_DB_PATH, CHROMA_COLLECTION_NAME, EMBEDDING_MODEL

print(f'Documents path : {RAG_DOCUMENTS_PATH}')
print(f'Chroma DB path : {CHROMA_DB_PATH}')
print(f'Collection     : {CHROMA_COLLECTION_NAME}')
print(f'Embedding model: {EMBEDDING_MODEL}')


## 1. Charger et inspecter les documents

In [ ]:
from src.rag.document_loader import DocumentLoader

loader = DocumentLoader()
docs = loader.load()

print(f'Nombre de chunks chargés : {len(docs)}')
if docs:
    sources = sorted(set(d.metadata.get('source', 'unknown') for d in docs))
    print(f'Sources uniques ({len(sources)}) :')
    for s in sources:
        count = sum(1 for d in docs if d.metadata.get('source') == s)
        print(f'  {Path(s).name:<50} {count} chunks')
    
    avg_size = sum(len(d.page_content) for d in docs) / len(docs)
    print(f'\nTaille moyenne des chunks : {avg_size:.0f} caractères')
    print(f'\nExemple de chunk (premier) :')
    print('---')
    print(docs[0].page_content[:400])
    print('---')
else:
    print('ERREUR: Aucun document chargé. Vérifiez que data/documents/ contient des fichiers .txt/.pdf/.docx')

## 2. Indexer dans Chroma

In [ ]:
from src.rag.vector_store import VectorStoreManager

assert len(docs) > 0, 'Pas de documents à indexer!'

print('Chargement du modèle d\'embeddings... (peut prendre quelques minutes au premier lancement)')
vsm = VectorStoreManager()

print(f'Indexation de {len(docs)} chunks...')
store = vsm.index_documents(docs)

print('\n✓ Indexation terminée!')
print(f'  Chroma DB créée dans : {CHROMA_DB_PATH}')

# Vérification
chroma_path = Path(CHROMA_DB_PATH)
if chroma_path.exists():
    files = list(chroma_path.rglob('*'))
    print(f'  Fichiers dans chroma_db : {len(files)}')
else:
    print('  ERREUR: Chroma DB non créée!')

## 3. Vérification avec 3 requêtes test

In [ ]:
from src.rag.retriever import Retriever

retriever = Retriever(vector_store_manager=vsm, top_k=3)

test_queries = [
    'À quoi sert un respirateur en réanimation ?',
    'Pourquoi les visites sont-elles limitées en réanimation ?',
    'Quel est le rôle des infirmiers en réanimation ?',
]

for query in test_queries:
    print(f'\n🔍 Requête : "{query}"')
    results = retriever.search(query)
    if results:
        for i, r in enumerate(results, 1):
            print(f'  [{i}] Source: {Path(r["source"]).name}')
            print(f'       Score: {r["score"]:.4f}')
            print(f'       Extrait: {r["text"][:120]}...')
    else:
        print('  ⚠️  Aucun résultat retourné!')

## 4. Résumé et validation

In [ ]:
print('=' * 55)
print('  RÉSUMÉ D\'INDEXATION — RAG ICU Chatbot')
print('=' * 55)
print(f'  Documents indexés : {len(sources)} fichiers')
print(f'  Chunks créés      : {len(docs)}')
print(f'  Chroma DB         : {CHROMA_DB_PATH}')
print(f'  Collection        : {CHROMA_COLLECTION_NAME}')
print(f'  Modèle embeddings : {EMBEDDING_MODEL}')
print('='  * 55)
print('  ✓ Le chatbot peut maintenant répondre à partir des')
print('    documents indexés. Lancer : streamlit run app.py')
print('=' * 55)